# 13.7 - Failure Handling

**Phase:** 13 - LangGraph / Stateful Workflows

**Status:** VERIFIED

---

## 1. What Are We Solving?

LLMs, APIs and databases fail. Explicit failure handling — try/except, retries, fallbacks, and error state — keeps one bad call from crashing the whole workflow.

## 2. Why Does This Matter?

## 3. Prerequisites

Units 13.1-13.6.

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Wrap risky work in `try/except` inside nodes
- Build a retry loop with exponential backoff and a max guard
- Route to fallback paths from error state
- Distinguish transient errors (retry) from permanent errors (fail fast)

## 5. Mental Model

Failure handling is a safety net: each node is a performer on a high wire; the net catches a fall and decides whether to retry, redirect, or stop the show. Error information lives in state so the router can make that decision.


## 6. Setup

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

GROQ_MODEL = os.environ.get("GROQ_MODEL", "openai/gpt-oss-20b")


def llm(prompt: str, model: str = GROQ_MODEL, temperature: float = 0.0) -> str:
    """One-shot Groq call with a deterministic mock fallback."""
    if not os.environ.get("GROQ_API_KEY"):
        return "mock: LangGraph is a stateful orchestration framework."
    try:
        from langchain_groq import ChatGroq
        chat = ChatGroq(model=model, temperature=temperature)
        return chat.invoke(prompt).content.strip()
    except Exception as e:  # network / quota / model errors -> never crash the notebook
        return f"[llm-error: {type(e).__name__}]"

print("Groq model:", GROQ_MODEL)
print("GROQ_API_KEY present:", bool(os.environ.get("GROQ_API_KEY")))


Groq model: openai/gpt-oss-20b
GROQ_API_KEY present: True


## 7. Retry Loop with Exponential Backoff

A flaky API call fails ~60% of the time. The conditional edge sends it back to `call` until it succeeds or `max_retries` runs out.

In [2]:
import time, random


class RobustState(TypedDict):
    job: str
    status: str | None
    attempt: int
    max_retries: int
    backoff: float


def call_api(state):
    attempt = state.get("attempt", 0) + 1
    if random.random() <= 0.6:
        return {"status": "ok", "attempt": attempt, "job": state["job"]}
    wait = min(state["backoff"] * (2 ** (attempt - 1)), 2.0)
    time.sleep(wait)
    return {"status": f"error (attempt {attempt})", "attempt": attempt, "job": state["job"]}


def fallback(state):
    return {"status": "fallback: served from cache", "job": state["job"]}


def route_after_attempt(state):
    if state["status"] == "ok":
        return "done"
    if state["attempt"] >= state["max_retries"]:
        return "fallback"
    return "retry"


g = StateGraph(RobustState)
g.add_node("call", call_api)
g.add_node("fallback", fallback)
g.add_edge(START, "call")
g.add_conditional_edges("call", route_after_attempt,
                         {"done": END, "retry": "call", "fallback": "fallback"})
g.add_edge("fallback", END)
random.seed(7)

out = g.compile().invoke({"job": "populate-dashboard", "status": None,
                         "attempt": 0, "max_retries": 3, "backoff": 0.2})
print(out)


{'job': 'populate-dashboard', 'status': 'ok', 'attempt': 1, 'max_retries': 3, 'backoff': 0.2}


## 8. In-Node Error Handling: Division

The simplest pattern catches the specific exception inside the node so the graph always reaches `END`.

In [3]:
class MathState(TypedDict):
    a: int
    b: int
    result: str | None


def divide(state):
    try:
        state["result"] = str(round(state["a"] / state["b"], 3))
    except ZeroDivisionError:
        state["result"] = "ERROR: division by zero"
    return state


g = StateGraph(MathState)
g.add_node("divide", divide)
g.add_edge(START, "divide")
g.add_edge("divide", END)
app = g.compile()
print(app.invoke({"a": 10, "b": 0, "result": None}))
print(app.invoke({"a": 10, "b": 3, "result": None}))


{'a': 10, 'b': 0, 'result': 'ERROR: division by zero'}
{'a': 10, 'b': 3, 'result': '3.333'}


## 9. Multi-Source Retrieval: try vector -> keyword -> cache

Track every source tried in state; if all fail, surface a helpful error instead of crashing.

In [4]:
class FetchState(TypedDict):
    query: str
    result: str | None
    error: str | None
    sources_tried: list


def fetch_source(name, query):
    if name == "vector_db":
        raise TimeoutError("vector db timed out")
    if name == "keyword":
        return f"keyword result for: {query}"
    return "cache result (stale but available)"


def retrieve(state):
    last_error = None
    for name in ("vector_db", "keyword", "cache"):
        state["sources_tried"] = state.get("sources_tried", []) + [name]
        try:
            state["result"] = fetch_source(name, state["query"])
            state["error"] = None
            return state
        except Exception as e:
            last_error = f"{name}: {e}"
    state["result"] = None
    state["error"] = "ALL SOURCES FAILED -> " + last_error
    return state


g = StateGraph(FetchState)
g.add_node("retrieve", retrieve)
g.add_edge(START, "retrieve")
g.add_edge("retrieve", END)
app = g.compile()

out = app.invoke({"query": "who won in 2026", "result": None, "error": None, "sources_tried": []})
print("sources tried:", out["sources_tried"])
print("result:", out["result"])
print("error:", out["error"])


sources tried: ['vector_db', 'keyword']
result: keyword result for: who won in 2026
error: None



## Common Mistakes

- **Return `None` from a node** — the graph silently drops the update. Always `return state`.
- **Mutating state in the router** — routers must be side-effect free.
- **Forgetting the terminal condition** — cycles run forever without an iteration guard.
- **Typo in a state key** — `TypedDict` catches it at compile time; plain dicts do not.

## Debugging

| Symptom | Likely Cause | Fix |
|---|---|---|
| `KeyError` on state | Field name mismatch | Match state keys to the `TypedDict` exactly |
| Graph won't compile | Node/referenced name typo | Check every string passed to `add_node`/`add_edge` |
| Node output ignored | Node returns `None` or a partial dict | Always `return state` (or a merge-able partial) |
| Infinite loop | No convergence guard | Add `max_steps` to state and check it in the router |
| Wrong branch taken | Router priority bug | Unit-test the router on every input variant |

## Best Practices

- Define all state fields upfront with defaults in a `TypedDict`.
- Keep node functions pure and focused: one responsibility each.
- Name nodes descriptively (`retrieve`, `generate`, not `step1`).
- Always add an iteration guard on loops.
- Inspect the graph with `app.get_graph().draw_mermaid()`.

## Hands-On Practice

1. **Basic:** Rerun the examples with new inputs; verify the trace.
2. **Guided:** Add a node that validates output before terminating.
3. **Independent:** Build a 3-step pipeline (fetch -> process -> summarize) with a retry node.
4. **Realistic:** Turn the example into a multi-department support agent.
5. **Challenge:** Save/load the state dict to JSON and resume the workflow from a checkpoint.

## Exit Criteria

- You can explain and build the concept from scratch.
- You can debug the associated failure modes.
- You know when to reach for this tool vs. a plain function.
